In [1]:
%load_ext autoreload
%autoreload 2

import functools
from math import pi
import os
from typing import NamedTuple


os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'


import matplotlib.pyplot as plt
import numpy as np
import tqdm.contrib
import xarray as xr

from fluxoniumcr import DATA_DIR
from fluxoniumcr.dressed_control_fluxonium import (
    create_driven_fluxonium,
    calculate_critical_amplitude,
    calculate_polarization_and_error,
    calculate_optimal_amplitude,
)
from fluxoniumcr.qubits.fluxonium import Fluxonium
from fluxoniumcr.utils import load_arguments

In [11]:
# parent_path = DATA_DIR/"polarization_spectrum2d/ELdivEJ=0.25,EC=1.0"
parent_path = DATA_DIR/"polarization_spectrum2d/ELdivEJ=0.10,EC=1.3"

argm = load_arguments(parent_path)

In [12]:
EC = argm.EC
ELdivEJ = argm.ELdivEJ

EJdivEC_data = argm.EJdivEC
drive_frequency_data = argm.frequencies

fluxonium_dim = argm.dim
fluxonium_cutoff = argm.cutoff
lookup_amps_unitless = argm.lookup_amplitudes
lookup_deg_tol = argm.lookup_degeneracy_tol


dataset = xr.Dataset(
    attrs=dict(
        ELdivEJ=ELdivEJ,
        EC=EC,
        lookup_degeneracy_tol=lookup_deg_tol,
    )
)

frequency = xr.DataArray(
    drive_frequency_data,
    dims=['frequency'],
    attrs=dict(
        units="Grad/s",
    )
)

EJdivEC = xr.DataArray(
    EJdivEC_data,
    dims=['EJdivEC'],
)

dataset['amplitude_unit'] = xr.DataArray(
    np.nan,
    coords=[EJdivEC],
    attrs=dict(
        units="Grad/s",
    )
)
dataset['amplitude'] = xr.DataArray(
    np.nan,
    coords=[EJdivEC, frequency],
    attrs=dict(
        units="Grad/s",
    )
)
dataset['p0'] = xr.DataArray(
    np.nan,
    coords=[EJdivEC, frequency],
)
dataset['p1'] = xr.DataArray(
    np.nan,
    coords=[EJdivEC, frequency],
)
dataset['error'] = xr.DataArray(
    np.nan,
    coords=[EJdivEC, frequency],
)

dataset_name = f"dataset_{len(dataset.EJdivEC)}x{len(dataset.frequency)}"

def load_dataset(suffix=""):
    return xr.load_dataset(parent_path/(dataset_name + suffix + ".hdf5"))

def save_dataset(dataset, suffix=""):
    dataset.to_netcdf(parent_path/(dataset_name + suffix + ".hdf5"))

In [13]:
# Load previous dataset
dataset = load_dataset()

In [15]:
from multiprocessing.pool import Pool


def doit(fx, drive_freq, relative_step_size, scale2=1, lookup_amps_override=None):    
    qubit_frequency = fx.eigenvalues[1] - fx.eigenvalues[0]
    
    if drive_freq < qubit_frequency/3:
        return (np.nan,)*4
    
    n_op = fx.get_operator('charge')
    Ω0 = qubit_frequency/abs(n_op[0, 1])

    floquet_basis = create_driven_fluxonium(
        fx,
        drive_freq,
    )
    
    if drive_freq < qubit_frequency:
        scale2 = qubit_frequency/drive_freq * 2
        scale2 = min(scale2, 10)
    
    if lookup_amps_override is not None:
        lookup_amps_unitless = lookup_amps_override
        
    floquet_basis.generate_lookup(
        Ω0 * lookup_amps_unitless * scale2,
        deg_tol=lookup_deg_tol,
    )
    amp, p0, p1, err = calculate_optimal_amplitude(
        fx,
        floquet_basis,
        step_size=relative_step_size * Ω0,
    )
    return amp, p0, p1, err


with Pool(processes=10) as pool:
    for EJdivEC in dataset.EJdivEC.data:
        print(EJdivEC)
        
        row = dataset.sel(EJdivEC=EJdivEC)
        
        EJ = EJdivEC * EC
        EL = ELdivEJ * EJ
        fx = Fluxonium(
            EJ=EJ,
            EC=EC,
            EL=EL,
            flux=0.5,
            dim=fluxonium_dim,
            cutoff=fluxonium_cutoff,
        )
        
        qubit_frequency = fx.eigenvalues[1] - fx.eigenvalues[0]
        n_op = fx.get_operator('charge')
        Ω0 = qubit_frequency/abs(n_op[0, 1])
        
        row['amplitude_unit'][()] = Ω0
        
        # Only calculate missing data.
        missing_frequency = row.frequency[row.error.isnull()].data.ravel()
        
        job_func = functools.partial(doit, fx, relative_step_size=0.005)
        
#         Smaller step size for filling in empty pixels.
#         job_func = functools.partial(
#             doit,
#             fx,
#             relative_step_size=0.001,
#             lookup_amps_override=np.linspace(0, 3, 151),
#         )

        for drive_freq, (amp, p0, p1, err) in tqdm.contrib.tzip(
                missing_frequency,
                pool.imap(
                    job_func,
                    missing_frequency,
                ),
        ):
            ds = row.loc[dict(frequency=drive_freq)]
            ds['amplitude'][()] = amp
            ds['p0'][()] = p0
            ds['p1'][()] = p1
            ds['error'][()] = err
        save_dataset(dataset, suffix="_autosave")

1.0


  0%|          | 0/2 [00:00<?, ?it/s]

1.01


  0%|          | 0/3 [00:00<?, ?it/s]

1.02


...

  0%|          | 0/9 [00:00<?, ?it/s]

1.42


  0%|          | 0/8 [00:00<?, ?it/s]

1.43


KeyboardInterrupt: 

In [ ]:
save_dataset(dataset)